In [2]:
from functools import lru_cache
from ecifp.utils import fetch_api_data
from ecifp.fp_sim import (get_ligand_ifp,
                          get_polymer_ifp,
                          get_ecifp_sim,
                          ECIFP)
from collections import defaultdict

In [3]:
@lru_cache
def get_bound_molecules(entry_id):
    entry_id = entry_id.lower()
    base_url = "https://www.ebi.ac.uk/pdbe/api/v2/pdb/bound_molecules/"
    url = f"{base_url}/{entry_id}"
    return fetch_api_data(url)

In [4]:
def get_bound_ccds(entry_id):
    bms = get_bound_molecules(entry_id)
    bound_ccds = []
    ccd_keys = ["chain_id", "author_residue_number", "chem_comp_id"]
    if not bms:
        return 
    for bm in bms[entry_id]:
        for ccd in bm['composition']['ligands']:
            bound_ccds.append(dict(
                filter(lambda item: item[0] in ccd_keys,
                       ccd.items()
                      )
            ))

    return bound_ccds  

In [5]:
@lru_cache
def get_bm_interactions(entry_id, bm_id):
    entry_id = entry_id.lower()
    base_url = "https://www.ebi.ac.uk/pdbe/api/v2/pdb/bound_molecule_interactions"
    url = f"{base_url}/{entry_id}/{bm_id}"
    return fetch_api_data(url)

In [6]:
@lru_cache
def get_ccd_interactions(entry_id, chain_id, seq_id):
    entry_id = entry_id.lower()
    base_url = "https://www.ebi.ac.uk/pdbe/api/v2/pdb/bound_ligand_interactions"
    url = f"{base_url}/{entry_id}/{chain_id}/{seq_id}"
    return fetch_api_data(url)

In [7]:
def get_intx_atoms(entry_id, chain_id, seq_id):
    intx = get_ccd_interactions(entry_id, chain_id, seq_id)
    ligand_id = intx[entry_id][0]['ligand']['chem_comp_id']
    bound_ccds = get_bound_ccds(entry_id)
    ccd_keys = ["chain_id", "author_residue_number", "chem_comp_id"]
    intx_to_remove = ["proximal","clash","vdw_clash","vdw"]
    intx_ligand_atoms = set()
    intx_polymer_atoms = defaultdict(set)
    for item in intx[entry_id][0]['interactions']:
        contact_types = [contact_type for contact_type in item['interaction_details'] if contact_type not in intx_to_remove]
        if not contact_types:
            continue
        intx_residue = {key:value for key,value in item['end'].items() if key in ccd_keys}
        if intx_residue in bound_ccds:
            continue
        if intx_residue['chem_comp_id'] in ['HOH','DOD']:
            continue
        intx_ligand_atoms = intx_ligand_atoms.union(set(item['ligand_atoms']))
        intx_polymer_atoms[item['end']['chem_comp_id']] = intx_polymer_atoms[item['end']['chem_comp_id']].union(set(item['end']['atom_names']))
    return (ligand_id, intx_ligand_atoms, intx_polymer_atoms)

In [8]:
def get_ecifp(entry_id, chain_id, seq_id, fp_radius=2, fp_size=1024):
    (ligand_id,intx_ligand_atoms, intx_polymer_atoms) = get_intx_atoms(entry_id, chain_id, seq_id)
    ligand_ifp = get_ligand_ifp(ligand_id, intx_ligand_atoms, fp_radius, fp_size)
    protein_ifp = get_polymer_ifp(intx_polymer_atoms, fp_radius, fp_size)
    return ECIFP(ligand_ifp, protein_ifp)

In [9]:
entry_id = "2c2b"
chain_id = 'A'
seq_id = 500
ecifp_1 = get_ecifp(entry_id, chain_id, seq_id)


In [10]:
entry_id = "2c2b"
chain_id = 'A'
seq_id = 501
ecifp_2 = get_ecifp(entry_id, chain_id, seq_id)

In [11]:
entry_id = "2c2b"
chain_id = 'B'
seq_id = 500
ecifp_3 = get_ecifp(entry_id, chain_id, seq_id)

In [12]:
entry_id = "2c2b"
chain_id = 'B'
seq_id = 501
ecifp_4 = get_ecifp(entry_id, chain_id, seq_id)

In [15]:
get_ecifp_sim(ecifp_1, ecifp_2,0.4, 0.6)

0.3646605791711311

In [16]:
get_ecifp_sim(ecifp_3, ecifp_4,0.4, 0.6)

0.38676183827972727

In [17]:
get_ecifp_sim(ecifp_1, ecifp_3, 0.4, 0.6)

0.9166239019255294

In [18]:
get_ecifp_sim(ecifp_2, ecifp_3, 0.4, 0.6)

0.38676183827972727

In [19]:
get_ecifp_sim(ecifp_1, ecifp_3,0.5, 0.5)

0.9218965030811009